In [14]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

In [15]:
DATE_CANDIDATE_COLUMNS = [
    "date",
    "Date",
    "order_date",
    "ship_date",
    "delivery_date",
    "return_date",
    "review_date",
    "signup_date",
    "snapshot_date",
    "start_date",
    "end_date",
]

KEY_LIKE_COLUMNS = {"zip", "date", "Date"}


def infer_relationship(left_df, left_key, right_df, right_key):
    """Suy ra nhãn quan hệ đơn giản dựa trên độ duy nhất của khóa và mức độ khớp."""
    left_non_null = left_df[left_key].dropna()
    right_non_null = right_df[right_key].dropna()

    left_unique = left_non_null.is_unique
    right_unique = right_non_null.is_unique

    if left_unique and right_unique:
        base = "1:1"
    elif left_unique and not right_unique:
        base = "1:n"
    elif not left_unique and right_unique:
        base = "n:1"
    else:
        base = "n:n"

    missing_refs = ~left_non_null.isin(set(right_non_null.unique()))
    unmatched_count = int(missing_refs.sum())
    if unmatched_count > 0:
        return base.replace("1", "0", 1)
    return base


def profile_foreign_key(dfs, left_table, left_key, right_table, right_key):
    """Kiểm tra một quan hệ khóa ngoại và báo cáo số lượng bản ghi không khớp."""
    if left_table not in dfs or right_table not in dfs:
        return {
            "left_table": left_table,
            "left_key": left_key,
            "right_table": right_table,
            "right_key": right_key,
            "checked_rows": None,
            "non_null_left_rows": None,
            "unmatched_rows": None,
            "match_rate_pct": None,
            "relationship": "not_checked",
            "note": "Thiếu một hoặc cả hai bảng.",
        }

    left_df = dfs[left_table]
    right_df = dfs[right_table]

    if left_key not in left_df.columns or right_key not in right_df.columns:
        return {
            "left_table": left_table,
            "left_key": left_key,
            "right_table": right_table,
            "right_key": right_key,
            "checked_rows": None,
            "non_null_left_rows": None,
            "unmatched_rows": None,
            "match_rate_pct": None,
            "relationship": "not_checked",
            "note": "Thiếu một hoặc cả hai cột khóa.",
        }

    left_non_null = left_df[left_key].dropna()
    right_values = set(right_df[right_key].dropna().unique())
    unmatched_mask = ~left_non_null.isin(right_values)
    unmatched_count = int(unmatched_mask.sum())
    match_rate = round((1 - unmatched_count / len(left_non_null)) * 100, 2) if len(left_non_null) else None
    relationship = infer_relationship(left_df, left_key, right_df, right_key)

    note = "Tất cả giá trị khóa khác null đều khớp."
    if unmatched_count > 0:
        note = f"Có {unmatched_count:,} dòng khác null trong {left_table}.{left_key} không khớp với {right_table}.{right_key}."

    return {
        "left_table": left_table,
        "left_key": left_key,
        "right_table": right_table,
        "right_key": right_key,
        "checked_rows": len(left_df),
        "non_null_left_rows": len(left_non_null),
        "unmatched_rows": unmatched_count,
        "match_rate_pct": match_rate,
        "relationship": relationship,
        "note": note,
    }

In [16]:
file_paths = [
    "dataset/customers.csv",
    "dataset/geography.csv",
    "dataset/inventory.csv",
    "dataset/orders.csv",
    "dataset/order_items.csv",
    "dataset/payments.csv",
    "dataset/products.csv",
    "dataset/promotions.csv",
    "dataset/returns.csv",
    "dataset/reviews.csv",
    "dataset/sales.csv",
    "dataset/sample_submission.csv",
    "dataset/shipments.csv",
    "dataset/submission.csv",
    "dataset/web_traffic.csv",
]


dfs = {}
load_summary = []

for file_path in file_paths:
    path = Path(file_path)
    table_name = path.stem

    # Chỉ parse các cột ngày thực sự có mặt trong header của file CSV.
    header_cols = pd.read_csv(path, nrows=0).columns.tolist()
    parse_dates = [col for col in DATE_CANDIDATE_COLUMNS if col in header_cols]

    df = pd.read_csv(path, parse_dates=parse_dates, low_memory=False)
    dfs[table_name] = df

    load_summary.append(
        {
            "table": table_name,
            "path": str(path),
            "rows": len(df),
            "columns": len(df.columns),
            "parsed_date_columns": ", ".join(parse_dates) if parse_dates else "-",
        }
    )

load_summary_df = pd.DataFrame(load_summary).sort_values("table").reset_index(drop=True)
display(load_summary_df)


,table,path,rows,columns,parsed_date_columns
0,customers,dataset\customers.csv,121930,7,signup_date
1,geography,dataset\geography.csv,39948,4,-
2,inventory,dataset\inventory.csv,60247,17,snapshot_date
3,order_items,dataset\order_items.csv,714669,7,-
4,orders,dataset\orders.csv,646945,8,order_date
5,payments,dataset\payments.csv,646945,4,-
6,products,dataset\products.csv,2412,8,-
7,promotions,dataset\promotions.csv,50,10,"start_date, end_date"
8,returns,dataset\returns.csv,39939,7,return_date
9,reviews,dataset\reviews.csv,113551,7,review_date


In [17]:
# Tổng quan các bảng
overview_rows = []

for table_name, df in dfs.items():
    display(Markdown(f"**{table_name}**"))
    print(f"Kích thước: {df.shape}")
    print(f"Danh sách cột ({len(df.columns)}): {list(df.columns)}")
    display(df.head(3))
    print("--------------------------------------------------------")

    overview_rows.append(
        {
            "table": table_name,
            "rows": len(df),
            "columns": len(df.columns),
            "column_names": ", ".join(df.columns),
        }
    )

overview_df = pd.DataFrame(overview_rows).sort_values(["rows", "table"], ascending=[False, True]).reset_index(drop=True)

display(Markdown("Bảng tổng hợp"))
display(overview_df)

**customers**

Kích thước: (121930, 7)
Danh sách cột (7): ['customer_id', 'zip', 'city', 'signup_date', 'gender', 'age_group', 'acquisition_channel']


,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search


--------------------------------------------------------


**geography**

...

Kích thước: (3652, 7)
Danh sách cột (7): ['date', 'sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source']


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct


--------------------------------------------------------


Bảng tổng hợp

,table,rows,columns,column_names
0,order_items,714669,7,"order_id, product_id, quantity, unit_price, discount_amount, promo_id, promo_id_2"
1,orders,646945,8,"order_id, order_date, customer_id, zip, order_status, payment_method, device_type, order_source"
2,payments,646945,4,"order_id, payment_method, payment_value, installments"
3,shipments,566067,4,"order_id, ship_date, delivery_date, shipping_fee"
4,customers,121930,7,"customer_id, zip, city, signup_date, gender, age_group, acquisition_channel"
5,reviews,113551,7,"review_id, order_id, product_id, customer_id, review_date, rating, review_title"
6,inventory,60247,17,"snapshot_date, product_id, stock_on_hand, units_received, units_sold, stockout_days, days_of_supply, fill_rate, stoc..."
7,geography,39948,4,"zip, city, region, district"
8,returns,39939,7,"return_id, order_id, product_id, return_date, return_reason, return_quantity, refund_amount"
9,sales,3833,3,"Date, Revenue, COGS"


In [18]:
# Khóa chính và khóa ngoại
pk_rows = []

for table_name, df in dfs.items():
    for col in df.columns:
        non_null = df[col].notna().sum()
        unique = df[col].nunique(dropna=True)
        uniqueness_ratio = round(unique / len(df), 4) if len(df) else None
        is_key_like = col.endswith("_id") or col in KEY_LIKE_COLUMNS
        is_candidate_pk = is_key_like and (non_null == len(df)) and (unique == len(df))

        if is_key_like:
            pk_rows.append(
                {
                    "table": table_name,
                    "column": col,
                    "non_null_rows": int(non_null),
                    "distinct_values": int(unique),
                    "row_count": len(df),
                    "uniqueness_ratio": uniqueness_ratio,
                    "key_like": is_key_like,
                    "candidate_primary_key": is_candidate_pk,
                }
            )

pk_df = pd.DataFrame(pk_rows).sort_values(
    ["candidate_primary_key", "table", "column"],
    ascending=[False, True, True],
).reset_index(drop=True)

display(Markdown("### Các cột có thể là khóa chính"))
display(pk_df[pk_df["candidate_primary_key"]].reset_index(drop=True))

display(Markdown("### Các cột giống khóa nhưng chưa đạt điều kiện khóa chính"))
display(pk_df[~pk_df["candidate_primary_key"]].reset_index(drop=True))

### Các cột có thể là khóa chính

,table,column,non_null_rows,distinct_values,row_count,uniqueness_ratio,key_like,candidate_primary_key
0,customers,customer_id,121930,121930,121930,1.0,True,True
1,geography,zip,39948,39948,39948,1.0,True,True
2,orders,order_id,646945,646945,646945,1.0,True,True
3,payments,order_id,646945,646945,646945,1.0,True,True
4,products,product_id,2412,2412,2412,1.0,True,True
5,promotions,promo_id,50,50,50,1.0,True,True
6,returns,return_id,39939,39939,39939,1.0,True,True
7,reviews,review_id,113551,113551,113551,1.0,True,True
8,sales,Date,3833,3833,3833,1.0,True,True
9,sample_submission,Date,548,548,548,1.0,True,True


### Các cột giống khóa nhưng chưa đạt điều kiện khóa chính

,table,column,non_null_rows,distinct_values,row_count,uniqueness_ratio,key_like,candidate_primary_key
0,customers,zip,121930,31491,121930,0.2583,True,False
1,inventory,product_id,60247,1624,60247,0.0270,True,False
2,order_items,order_id,714669,646945,714669,0.9052,True,False
3,order_items,product_id,714669,1598,714669,0.0022,True,False
4,order_items,promo_id,276316,50,714669,0.0001,True,False
5,orders,customer_id,646945,90246,646945,0.1395,True,False
6,orders,zip,646945,29932,646945,0.0463,True,False
7,returns,order_id,39939,36062,39939,0.9029,True,False
8,returns,product_id,39939,1286,39939,0.0322,True,False
9,reviews,customer_id,113551,48676,113551,0.4287,True,False


In [19]:
fk_checks = [
    ("customers", "zip", "geography", "zip"),
    ("orders", "customer_id", "customers", "customer_id"),
    ("orders", "zip", "geography", "zip"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "promo_id", "promotions", "promo_id"),
    ("order_items", "promo_id_2", "promotions", "promo_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("shipments", "order_id", "orders", "order_id"),
    ("returns", "order_id", "orders", "order_id"),
    ("returns", "product_id", "products", "product_id"),
    ("reviews", "order_id", "orders", "order_id"),
    ("reviews", "product_id", "products", "product_id"),
    ("reviews", "customer_id", "customers", "customer_id"),
    ("inventory", "product_id", "products", "product_id"),
]

fk_df = pd.DataFrame(
    [
        profile_foreign_key(dfs, left_table, left_key, right_table, right_key)
        for left_table, left_key, right_table, right_key in fk_checks
    ]
)

fk_df["note"] = fk_df["note"].replace("Tất cả giá trị khóa khác null đều khớp.", "")

display(Markdown("### Kiểm tra khóa ngoại"))
display(fk_df)

display(Markdown("### Các quan hệ có mismatch"))
display(fk_df[fk_df["unmatched_rows"].fillna(0) > 0].reset_index(drop=True))


### Kiểm tra khóa ngoại

,left_table,left_key,right_table,right_key,checked_rows,non_null_left_rows,unmatched_rows,match_rate_pct,relationship,note
0,customers,zip,geography,zip,121930,121930,0,100.0,n:1,
1,orders,customer_id,customers,customer_id,646945,646945,0,100.0,n:1,
2,orders,zip,geography,zip,646945,646945,0,100.0,n:1,
3,order_items,order_id,orders,order_id,714669,714669,0,100.0,n:1,
4,order_items,product_id,products,product_id,714669,714669,0,100.0,n:1,
5,order_items,promo_id,promotions,promo_id,714669,276316,0,100.0,n:1,
6,order_items,promo_id_2,promotions,promo_id,714669,206,0,100.0,n:1,
7,payments,order_id,orders,order_id,646945,646945,0,100.0,1:1,
8,shipments,order_id,orders,order_id,566067,566067,0,100.0,1:1,
9,returns,order_id,orders,order_id,39939,39939,0,100.0,n:1,


### Các quan hệ có mismatch

,left_table,left_key,right_table,right_key,checked_rows,non_null_left_rows,unmatched_rows,match_rate_pct,relationship,note


In [20]:
# Giá  trị thiếu
missing_rows = []

for table_name, df in dfs.items():
    missing_count = df.isna().sum()
    missing_pct = (missing_count / len(df) * 100).round(2) if len(df) else 0

    for col in df.columns:
        if missing_count[col] > 0:
            missing_rows.append(
                {
                    "table": table_name,
                    "column": col,
                    "missing_count": int(missing_count[col]),
                    "missing_pct": float(missing_pct[col]),
                }
            )

missing_df = pd.DataFrame(missing_rows).sort_values(
    ["missing_pct", "missing_count", "table", "column"],
    ascending=[False, False, True, True],
).reset_index(drop=True)

display(Markdown("### Các cột có giá trị thiếu"))
display(missing_df)

display(Markdown("### Các cột thiếu đáng chú ý nhất"))
display(missing_df.head(10))


### Các cột có giá trị thiếu

,table,column,missing_count,missing_pct
0,order_items,promo_id_2,714463,99.97
1,promotions,applicable_category,40,80.00
2,order_items,promo_id,438353,61.34


### Các cột thiếu đáng chú ý nhất

,table,column,missing_count,missing_pct
0,order_items,promo_id_2,714463,99.97
1,promotions,applicable_category,40,80.00
2,order_items,promo_id,438353,61.34


In [21]:
# Khoảng thời gian dữ liệu
date_rows = []

for table_name, df in dfs.items():
    date_cols = [col for col in df.columns if col in DATE_CANDIDATE_COLUMNS or pd.api.types.is_datetime64_any_dtype(df[col])]

    for col in date_cols:
        series = pd.to_datetime(df[col], errors="coerce").dropna()
        if len(series) == 0:
            continue

        date_rows.append(
            {
                "table": table_name,
                "date_column": col,
                "min_date": series.min(),
                "max_date": series.max(),
                "distinct_dates": int(series.nunique()),
            }
        )

date_ranges_df = pd.DataFrame(date_rows).sort_values(["table", "date_column"]).reset_index(drop=True)
display(date_ranges_df)

,table,date_column,min_date,max_date,distinct_dates
0,customers,signup_date,2012-01-17,2022-12-31,3941
1,inventory,snapshot_date,2012-07-31,2022-12-31,126
2,orders,order_date,2012-07-04,2022-12-31,3833
3,promotions,end_date,2013-03-01,2022-12-31,50
4,promotions,start_date,2013-01-31,2022-11-18,50
5,returns,return_date,2012-07-11,2022-12-31,3806
6,reviews,review_date,2012-07-10,2022-12-31,3825
7,sales,Date,2012-07-04,2022-12-31,3833
8,sample_submission,Date,2023-01-01,2024-07-01,548
9,shipments,delivery_date,2012-07-06,2022-12-31,3831


In [22]:
# Sơ đồ join
join_map_rows = []

for left_table, left_key, right_table, right_key in fk_checks:
    result = profile_foreign_key(dfs, left_table, left_key, right_table, right_key)
    join_map_rows.append(
        {
            "left_table": result["left_table"],
            "left_key": result["left_key"],
            "right_table": result["right_table"],
            "right_key": result["right_key"],
            "relationship_note": result["relationship"],
            "validation_note": result["note"],
        }
    )

join_map_df = pd.DataFrame(join_map_rows)
display(join_map_df)

,left_table,left_key,right_table,right_key,relationship_note,validation_note
0,customers,zip,geography,zip,n:1,Tất cả giá trị khóa khác null đều khớp.
1,orders,customer_id,customers,customer_id,n:1,Tất cả giá trị khóa khác null đều khớp.
2,orders,zip,geography,zip,n:1,Tất cả giá trị khóa khác null đều khớp.
3,order_items,order_id,orders,order_id,n:1,Tất cả giá trị khóa khác null đều khớp.
4,order_items,product_id,products,product_id,n:1,Tất cả giá trị khóa khác null đều khớp.
5,order_items,promo_id,promotions,promo_id,n:1,Tất cả giá trị khóa khác null đều khớp.
6,order_items,promo_id_2,promotions,promo_id,n:1,Tất cả giá trị khóa khác null đều khớp.
7,payments,order_id,orders,order_id,1:1,Tất cả giá trị khóa khác null đều khớp.
8,shipments,order_id,orders,order_id,1:1,Tất cả giá trị khóa khác null đều khớp.
9,returns,order_id,orders,order_id,n:1,Tất cả giá trị khóa khác null đều khớp.


In [23]:
# Bảng nào hỗ trợ MCQ/ EDA/ Forecasting
use_case_rows = [
    {
        "table": "customers",
        "MCQ": True,
        "EDA": True,
        "Forecasting": False,
        "reason": "Bảng khách hàng phù hợp cho câu hỏi mô tả và phân khúc, nhưng không phải mục tiêu dự báo chuỗi thời gian trực tiếp.",
    },
    {
        "table": "geography",
        "MCQ": True,
        "EDA": True,
        "Forecasting": False,
        "reason": "Bảng địa lý là bảng tham chiếu hữu ích cho join, tổng hợp theo khu vực và bổ sung ngữ cảnh kinh doanh.",
    },
    {
        "table": "inventory",
        "MCQ": True,
        "EDA": True,
        "Forecasting": True,
        "reason": "Bảng tồn kho có cấu trúc snapshot theo tháng và các chỉ số vận hành, phù hợp cho dự báo tồn kho hoặc nhu cầu.",
    },
    {
        "table": "orders",
        "MCQ": True,
        "EDA": True,
        "Forecasting": True,
        "reason": "Bảng đơn hàng là bảng giao dịch lõi, có mốc thời gian và liên kết khách hàng để phục vụ cả phân tích lẫn dự báo.",
    },
    {
        "table": "order_items",
        "MCQ": True,
        "EDA": True,
        "Forecasting": True,
        "reason": "Bảng chi tiết đơn hàng hỗ trợ phân tích ở mức sản phẩm và có thể tổng hợp thành các mục tiêu dự báo.",
    },
    {
        "table": "payments",
        "MCQ": True,
        "EDA": True,
        "Forecasting": False,
        "reason": "Bảng thanh toán hữu ích cho câu hỏi kinh doanh về phương thức và giá trị thanh toán, nhưng thường không phải mục tiêu dự báo độc lập.",
    },
    {
        "table": "products",
        "MCQ": True,
        "EDA": True,
        "Forecasting": False,
        "reason": "Bảng sản phẩm là bảng chiều giúp làm giàu phân tích theo danh mục, phân khúc, kích cỡ và giá.",
    },
    {
        "table": "promotions",
        "MCQ": True,
        "EDA": True,
        "Forecasting": False,
        "reason": "Bảng khuyến mãi cung cấp metadata chiến dịch, hữu ích để giải thích biến động bán hàng và mức độ sử dụng promo.",
    },
    {
        "table": "returns",
        "MCQ": True,
        "EDA": True,
        "Forecasting": True,
        "reason": "Bảng trả hàng phù hợp cho phân tích chất lượng và có thể dùng để dự báo lượng trả hàng sau khi tổng hợp theo thời gian.",
    },
    {
        "table": "reviews",
        "MCQ": True,
        "EDA": True,
        "Forecasting": False,
        "reason": "Bảng đánh giá phù hợp cho phân tích mức độ hài lòng và chất lượng sản phẩm, nhưng không phải mục tiêu dự báo chính.",
    },
    {
        "table": "sales",
        "MCQ": True,
        "EDA": True,
        "Forecasting": True,
        "reason": "Bảng doanh thu là chuỗi thời gian theo ngày gọn nhất và là mục tiêu trực tiếp rõ ràng nhất cho bài toán forecasting.",
    },
    {
        "table": "sample_submission",
        "MCQ": False,
        "EDA": False,
        "Forecasting": True,
        "reason": "Bảng sample submission chủ yếu định nghĩa format đầu ra dự báo, không phản ánh hành vi kinh doanh.",
    },
    {
        "table": "shipments",
        "MCQ": True,
        "EDA": True,
        "Forecasting": True,
        "reason": "Bảng giao hàng hỗ trợ phân tích hiệu suất vận chuyển và có thể dùng cho dự báo số lượng giao hoặc lead time.",
    },
    {
        "table": "submission",
        "MCQ": False,
        "EDA": False,
        "Forecasting": True,
        "reason": "Bảng submission là file đầu ra cho kết quả dự báo, không phải bảng nguồn để phân tích.",
    },
    {
        "table": "web_traffic",
        "MCQ": True,
        "EDA": True,
        "Forecasting": True,
        "reason": "Bảng web traffic là chuỗi thời gian hành vi theo ngày, hữu ích cho phân tích xu hướng và dự báo tín hiệu nhu cầu.",
    },
]

use_case_df = pd.DataFrame(use_case_rows).sort_values("table").reset_index(drop=True)
display(use_case_df)

,table,MCQ,EDA,Forecasting,reason
0,customers,True,True,False,"Bảng khách hàng phù hợp cho câu hỏi mô tả và phân khúc, nhưng không phải mục tiêu dự báo chuỗi thời gian trực tiếp."
1,geography,True,True,False,"Bảng địa lý là bảng tham chiếu hữu ích cho join, tổng hợp theo khu vực và bổ sung ngữ cảnh kinh doanh."
2,inventory,True,True,True,"Bảng tồn kho có cấu trúc snapshot theo tháng và các chỉ số vận hành, phù hợp cho dự báo tồn kho hoặc nhu cầu."
3,order_items,True,True,True,Bảng chi tiết đơn hàng hỗ trợ phân tích ở mức sản phẩm và có thể tổng hợp thành các mục tiêu dự báo.
4,orders,True,True,True,"Bảng đơn hàng là bảng giao dịch lõi, có mốc thời gian và liên kết khách hàng để phục vụ cả phân tích lẫn dự báo."
5,payments,True,True,False,"Bảng thanh toán hữu ích cho câu hỏi kinh doanh về phương thức và giá trị thanh toán, nhưng thường không phải mục tiê..."
6,products,True,True,False,"Bảng sản phẩm là bảng chiều giúp làm giàu phân tích theo danh mục, phân khúc, kích cỡ và giá."
7,promotions,True,True,False,"Bảng khuyến mãi cung cấp metadata chiến dịch, hữu ích để giải thích biến động bán hàng và mức độ sử dụng promo."
8,returns,True,True,True,Bảng trả hàng phù hợp cho phân tích chất lượng và có thể dùng để dự báo lượng trả hàng sau khi tổng hợp theo thời gian.
9,reviews,True,True,False,"Bảng đánh giá phù hợp cho phân tích mức độ hài lòng và chất lượng sản phẩm, nhưng không phải mục tiêu dự báo chính."


In [24]:
# Kết luận
candidate_pk_count = int(pk_df["candidate_primary_key"].sum()) if not pk_df.empty else 0
fk_mismatch_count = int((fk_df["unmatched_rows"].fillna(0) > 0).sum()) if not fk_df.empty else 0
tables_with_missing = int(missing_df["table"].nunique()) if not missing_df.empty else 0
tables_with_dates = int(date_ranges_df["table"].nunique()) if not date_ranges_df.empty else 0

takeaways = [
    f"- Đã load {len(dfs)} bảng vào dictionary `dfs`.",
    f"- Tìm thấy {candidate_pk_count} cột có thể xem là khóa chính dựa trên tính duy nhất tuyệt đối.",
    f"- Phát hiện {fk_mismatch_count} quan hệ khóa ngoại có ít nhất một mismatch.",
    f"- Có giá trị thiếu trong {tables_with_missing} bảng.",
    f"- Có cột ngày trong {tables_with_dates} bảng, đủ để hỗ trợ EDA theo thời gian và chuẩn bị cho forecasting.",
]

display(Markdown("\n".join(takeaways)))


- Đã load 15 bảng vào dictionary `dfs`.
- Tìm thấy 13 cột có thể xem là khóa chính dựa trên tính duy nhất tuyệt đối.
- Phát hiện 0 quan hệ khóa ngoại có ít nhất một mismatch.
- Có giá trị thiếu trong 2 bảng.
- Có cột ngày trong 11 bảng, đủ để hỗ trợ EDA theo thời gian và chuẩn bị cho forecasting.